# 🔴 Solution: Grouped Query Attention（参考版）

## 📋 题目描述

**难度：** Hard

实现分组查询注意力（GQA）。

GQA 使用比查询头更少的 KV 头，每个 KV 头在一组查询头之间共享，在保持质量的同时减少 KV 缓存大小。

**签名:** `GroupQueryAttention(d_model, num_heads, num_kv_heads)`

**前向传播:** `forward(x) -> Tensor`
- `x` — 输入张量 (B, S, d_model)

**返回:** 注意力输出 (B, S, d_model)

**约束:**
- W_k/W_v 投影到 `num_kv_heads * d_k` 维
- 使用 `repeat_interleave` 扩展 KV 头以匹配查询头数
- 当 `num_kv_heads == num_heads` 时退化为标准 MHA

**提示：** 1. `W_q` → `(B,H,S,d_k)`，`W_k`/`W_v` → `(B,KV,S,d_k)`
2. `K = K.repeat_interleave(H//KV, dim=1)` 扩展 KV 头
3. 缩放点积注意力 → reshape 为 `(B,S,d_model)`


In [ ]:
import torch
import math

In [ ]:
# ✅ SOLUTION

def grouped_query_attention(x: torch.Tensor, W_q: torch.Tensor, W_k: torch.Tensor, W_v: torch.Tensor, W_o: torch.Tensor, num_heads: int, num_kv_heads: int, mask: torch.Tensor = None) -> torch.Tensor:
    # Grouped Query Attention (GQA)
    # MHA: 每个 head 有自己的 Q, K, V
    # GQA: 多个 query head 共享一组 K, V（减少 KV cache 大小）
    # 例如 num_heads=8, num_kv_heads=2 → 每 4 个 query head 共享 1 组 KV

    batch_size, seq_len, d_model = x.shape
    d_k = d_model // num_heads

    # ---- Step 1: 投影 Q, K, V ----
    Q = x @ W_q  # [B, S, d_model]
    K = x @ W_k  # [B, S, num_kv_heads * d_k]
    V = x @ W_v  # [B, S, num_kv_heads * d_k]

    # ---- Step 2: 拆分并 reshape ----
    Q = Q.view(batch_size, seq_len, num_heads, d_k).transpose(1, 2)  # [B, H, S, d_k]
    K = K.view(batch_size, seq_len, num_kv_heads, d_k).transpose(1, 2)  # [B, G, S, d_k]
    V = V.view(batch_size, seq_len, num_kv_heads, d_k).transpose(1, 2)  # [B, G, S, d_k]

    # ---- Step 3: 扩展 KV 以匹配 Q 的 head 数 ----
    # repeat_interleave：将每个 KV head 复制 num_heads//num_kv_heads 次
    # 例如 2 KV heads → 8 KV heads（每个复制 4 次）
    n_rep = num_heads // num_kv_heads
    K = K.repeat_interleave(n_rep, dim=1)
    V = V.repeat_interleave(n_rep, dim=1)

    # ---- Step 4: 标准注意力计算 ----
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    attn = torch.softmax(scores, dim=-1)
    out = attn @ V

    # ---- Step 5: 合并多头 + 输出投影 ----
    out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)
    return out @ W_o

In [ ]:
gqa = GroupQueryAttention(32, 8, 2)
print('Output:', gqa.forward(torch.randn(1, 4, 32)).shape)

## 📝 核心思路总结

1. **GQA 核心**：多个 query head 共享 KV head，减少 KV cache
2. **repeat_interleave**：扩展 KV 以匹配 query head 数
3. **GQA = MHA + MQA 的折中**：平衡效果和效率
4. **推理优化**：KV cache 减少 num_heads/num_kv_heads 倍